# Inference & model comparison overview
This notebook loads trained models and the preprocessed datasets, evaluates models via cross-validation, saves ROC and loss plots, updates a run history, and writes submission CSVs for Kaggle.

In [24]:
## End-to-end sklearn Pipeline: transform test like train and create submission
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, roc_curve, log_loss
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt



PROCESS_PATH = '../result/processed'
SUBMISSION_PATH = '../result/submission'
MODEL_PATH = '../result/model'
PIC_PATH = '../result/pic'
SEED = 42
timestamp = input('Enter timestamp string: ')

train = pd.read_csv(f'{PROCESS_PATH}/titanic_train_preprocessed.csv')
test = pd.read_csv(f'{PROCESS_PATH}/titanic_test_preprocessed.csv')

## Setup: paths, timestamp, and data

This cell prepares the inference and model-comparison environment. We define the canonical result paths used across notebooks—`PROCESS_PATH` for preprocessed matrices and metadata, `MODEL_PATH` for serialized estimators, and `PIC_PATH` for plots. To ensure we evaluate and submit with the intended artifacts, we request a `timestamp` string: this is the same identifier printed at the end of the training notebook and embedded into filenames (e.g., `randomForest_Base_<timestamp>.pkl`). Using a timestamp avoids accidental mismatches between models and features from different runs, a common source of silent errors.

We then load the preprocessed train and test matrices from `../result/processed/` produced by `code.ipynb`. At this point, both matrices should share the same columns (except the training matrix has the label `Survived`). If loading fails, re-run the preprocessing notebook or verify path consistency. Small sanity checks (e.g., verifying `Survived` is present only in train and inspecting the number of columns) can catch drift early. Maintaining strict alignment of feature columns is essential because the saved models expect a specific feature order and presence; we’ll handle alignment rigorously in later cells when unpacking models’ stored feature lists.

Finally, we import common evaluation utilities (accuracy, ROC AUC, F1, ROC curves, log loss) and CV splitters. Even though this notebook focuses on inference and comparison, re-computing cross-validated metrics on the training set is valuable to confirm performance and to create consistent ROC and loss diagnostic plots for each loaded model. If your environment changes or new packages are added, ensure versions are compatible with the pickled models to prevent deserialization issues.

In [25]:
# Fit on full train and predict on test
randomForest_base_pkl = joblib.load(f"{MODEL_PATH}/randomForest_Base_{timestamp}.pkl")
randomForest_best_all_features_pkl = joblib.load(f"{MODEL_PATH}/rf_best_all_features_{timestamp}.pkl")
mi_randomForest_25_features_pkl = joblib.load(f"{MODEL_PATH}/mi_randomForest_25_{timestamp}_features.pkl")

y_full = train['Survived'].astype(int)

# Optionally extract (model, features) if a pickle stores a dict/tuple
def unpack(obj):
    # supports dict {'model': ..., 'features': [...]}, tuple (model, features), or plain model
    if isinstance(obj, dict):
        return obj.get('model', obj), obj.get('features')
    if isinstance(obj, tuple) and len(obj) == 2:
        return obj[0], obj[1]
    return obj, None

rf_base_model, rf_base_feats = unpack(randomForest_base_pkl)
rf_best_model, rf_best_feats = unpack(randomForest_best_all_features_pkl)
mi_model, mi_feats_25 = unpack(mi_randomForest_25_features_pkl)

all_features = [c for c in train.columns if c != 'Survived']

models = [
    ("randomForest_base", rf_base_model, rf_base_feats or all_features),
    ("rf_best_all_features", rf_best_model, rf_best_feats or all_features),
    ("mi_randomForest_25_features", mi_model, mi_feats_25 or all_features),
]


FileNotFoundError: [Errno 2] No such file or directory: '../result/model/randomForest_Base_.pkl'

In [ ]:
# score each model
X= train.drop(columns=['Survived', 'PassengerId'])
y= train['Survived'].astype(int)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

accs, aucs, save_feature = {}, {}, {}
roc_points, losses = {}, {}

def get_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        # convert scores to (0,1) for plotting/log loss only
        return 1.0 / (1.0 + np.exp(-scores))
    else:
        # fallback to hard predictions (0/1)
        return model.predict(X)

for name, model, features in models:
    X_model = X[features]
    for fold_idx, (train_idx, valid_idx) in enumerate(kfold.split(X_model, y), start=1):
        X_train, X_valid = X_model.iloc[train_idx], X_model.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_valid)
        acc = accuracy_score(y_valid, y_pred)
        # probability scores for ROC and log loss
        try:
            y_score = get_proba(model, X_valid)
        except Exception:
            y_score = model.predict(X_valid)
        fpr, tpr, _ = roc_curve(y_valid, y_score)
        roc_points.setdefault(name, []).append((fpr, tpr))
        # clip probabilities to avoid log(0)
        losses.setdefault(name, []).append(log_loss(y_valid, np.clip(y_score, 1e-15, 1 - 1e-15)))
        auc = roc_auc_score(y_valid, y_score if y_score is not None else y_pred)
        accs.setdefault(name, []).append(acc)
        aucs.setdefault(name, []).append(auc)
    save_feature.setdefault(name, []).append(features)
    print(f"Model: {name} - CV Accuracy: {sum(accs[name])/len(accs[name]):.4f}, CV ROC AUC: {sum(aucs[name])/len(aucs[name]):.4f}")
    accs[name + '_mean'] = sum(accs[name]) / len(accs[name])
    aucs[name + '_mean'] = sum(aucs[name]) / len(aucs[name])

    # Save ROC curve and loss plot for this model
    os.makedirs(f'{PIC_PATH}/score', exist_ok=True)
    # ROC curve (mean across folds with per-fold traces)
    mean_fpr = np.linspace(0, 1, 100)
    tprs = []
    plt.figure()
    for fpr, tpr in roc_points.get(name, []):
        plt.plot(fpr, tpr, color='gray', alpha=0.2)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    if tprs:
        tprs = np.array(tprs)
        mean_tpr = tprs.mean(axis=0)
        std_tpr = tprs.std(axis=0)
        auc_mean_local = sum(aucs[name]) / len(aucs[name]) if aucs.get(name) else None
        plt.plot(mean_fpr, mean_tpr, color='blue', label=(f'Mean ROC (AUC={auc_mean_local:.3f})' if auc_mean_local is not None else 'Mean ROC'))
        tpr_upper = np.minimum(mean_tpr + std_tpr, 1)
        tpr_lower = np.maximum(mean_tpr - std_tpr, 0)
        plt.fill_between(mean_fpr, tpr_lower, tpr_upper, color='blue', alpha=0.1, label='±1 std')
    plt.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1, label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {name}')
    plt.legend(loc='lower right')
    roc_file = f"{PIC_PATH}/score/{name}_roc_{timestamp}.png"
    plt.savefig(roc_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved ROC plot to {roc_file}")

    # Loss plot (log loss per fold)
    plt.figure()
    fold_losses = losses.get(name, [])
    if fold_losses:
        plt.plot(range(1, len(fold_losses)+1), fold_losses, marker='o', label='Validation log loss')
        mean_loss = float(sum(fold_losses) / len(fold_losses))
        plt.axhline(mean_loss, color='red', linestyle='--', label=f'Mean={mean_loss:.4f}')
    plt.xlabel('Fold')
    plt.ylabel('Log loss')
    plt.title(f'Validation Log Loss per Fold - {name}')
    plt.legend()
    loss_file = f"{PIC_PATH}/score/{name}_loss_{timestamp}.png"
    plt.savefig(loss_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved loss plot to {loss_file}")


Model: randomForest_base - CV Accuracy: 0.8216, CV ROC AUC: 0.8726
Saved ROC plot to ../result/pic/score/randomForest_base_roc_20251102-030411.png
Saved loss plot to ../result/pic/score/randomForest_base_loss_20251102-030411.png
Saved loss plot to ../result/pic/score/randomForest_base_loss_20251102-030411.png
Model: rf_best_all_features - CV Accuracy: 0.8384, CV ROC AUC: 0.8830
Saved ROC plot to ../result/pic/score/rf_best_all_features_roc_20251102-030411.png
Saved loss plot to ../result/pic/score/rf_best_all_features_loss_20251102-030411.png
Model: rf_best_all_features - CV Accuracy: 0.8384, CV ROC AUC: 0.8830
Saved ROC plot to ../result/pic/score/rf_best_all_features_roc_20251102-030411.png
Saved loss plot to ../result/pic/score/rf_best_all_features_loss_20251102-030411.png
Model: mi_randomForest_25_features - CV Accuracy: 0.8384, CV ROC AUC: 0.8830
Saved ROC plot to ../result/pic/score/mi_randomForest_25_features_roc_20251102-030411.png
Saved loss plot to ../result/pic/score/mi_rand

In [ ]:
# Save accuracies and AUCs to JSON
import json
acc_auc_path = f"{PROCESS_PATH}/accs-aucs.json"

try:
    with open(acc_auc_path, "r") as f:
        content = f.read().strip()
        data = json.loads(content) if content else []
except (FileNotFoundError, json.JSONDecodeError) as e:
    print(f"Warning: initializing {acc_auc_path}. Reason: {e}")
    data = []

print(data)
data.append({"time": timestamp, "accs": accs, "aucs": aucs, "features": save_feature})

with open(acc_auc_path, "w") as f:
    json.dump(data, f)


[{'time': '20251031-021725', 'accs': {'randomForest_base': [0.7555555555555555, 0.8089887640449438, 0.8202247191011236, 0.7415730337078652, 0.7865168539325843, 0.8089887640449438, 0.8539325842696629, 0.7865168539325843, 0.8202247191011236, 0.7752808988764045], 'randomForest_base_mean': 0.7957802746566791, 'rf_best_all_features': [0.8333333333333334, 0.8089887640449438, 0.8089887640449438, 0.8651685393258427, 0.8202247191011236, 0.8764044943820225, 0.8651685393258427, 0.8202247191011236, 0.8539325842696629, 0.8314606741573034], 'rf_best_all_features_mean': 0.8383895131086142, 'mi_randomForest_25_features': [0.8333333333333334, 0.8202247191011236, 0.8089887640449438, 0.8651685393258427, 0.8202247191011236, 0.8651685393258427, 0.8651685393258427, 0.8089887640449438, 0.8651685393258427, 0.8202247191011236], 'mi_randomForest_25_features_mean': 0.8372659176029963}, 'aucs': {'randomForest_base': [0.7584415584415584, 0.7893048128342247, 0.8040106951871658, 0.7347593582887701, 0.765508021390374

In [ ]:
os.makedirs(SUBMISSION_PATH, exist_ok=True)
# Create submissions
for name, model, features in models:
    X_tr = train[features]
    X_te = test[features]

    # fit only if not already fitted
    if not hasattr(model, "classes_"):
        model.fit(X_tr, y_full)

    test_pred = model.predict(X_te)
    sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": test_pred.astype(int)})
    sub_path = f"{SUBMISSION_PATH}/submission_{name}_{timestamp}.csv"
    sub.to_csv(sub_path, index=False)
    print(f"Saved submission to {sub_path}")


Saved submission to ../result/processed/submission_randomForest_base_20251102-030411.csv
Saved submission to ../result/processed/submission_rf_best_all_features_20251102-030411.csv
Saved submission to ../result/processed/submission_mi_randomForest_25_features_20251102-030411.csv


In [ ]:
# Compare historical runs: find best mean accuracy and its features
import json, os

acc_auc_path = f"{PROCESS_PATH}/accs-aucs.json"
try:
    with open(acc_auc_path, "r") as f:
        content = f.read().strip()
        history = json.loads(content) if content else []
except (FileNotFoundError, json.JSONDecodeError) as e:
    print(f"Cannot read {acc_auc_path}: {e}")
    history = []

if not history:
    print("No history found in accs-aucs.json")
else:
    best = {
        "mean_acc": -1.0,
        "model": None,
        "auc_mean": None,
        "features": None,
        "time": None,
        "index": None,
    }
    for idx, entry in enumerate(history):
        accs = entry.get("accs", {})
        aucs = entry.get("aucs", {})
        feats = entry.get("features", {})
        for k, v in accs.items():
            if k.endswith("_mean") and isinstance(v, (int, float)):
                model = k[:-5]  # strip suffix
                mean_acc = float(v)
                if mean_acc > best["mean_acc"]:
                    # pick the last recorded features list for this model if present
                    fval = feats.get(model)
                    features_list = fval[-1] if isinstance(fval, list) and fval else None
                    best.update({
                        "mean_acc": mean_acc,
                        "model": model,
                        "auc_mean": float(aucs.get(model + "_mean")) if isinstance(aucs.get(model + "_mean"), (int, float)) else None,
                        "features": features_list,
                        "time": entry.get("time"),
                        "index": idx,
                    })
    print("Best run across history:")
    summary = {
        "time": best["time"],
        "run_index": best["index"],
        "model": best["model"],
        "mean_accuracy": round(best["mean_acc"], 4) if best["mean_acc"] is not None else None,
        "mean_auc": round(best["auc_mean"], 4) if best["auc_mean"] is not None else None,
        "n_features": len(best["features"]) if isinstance(best["features"], list) else None,
        "features": best["features"],
    }
    print(summary)

Best run across history:
{'time': '20251031-092749', 'run_index': 10, 'model': 'mi_randomForest_25_features', 'mean_accuracy': 0.8462, 'mean_auc': 0.829, 'n_features': 12, 'features': ['Title_Mr', 'Fare', 'Title_Miss', 'FamilySize', 'Age', 'Pclass', 'SibSp', 'Title_Mrs', 'IsAlone', 'Title_Rare', 'Embarked_Q', 'Embarked_S']}
